# 4.2 RLDA — 회귀 기반 프로파일링과 정보량 추정

## 이 노트북이 답하는 질문

4.0 의 LDA 는 클래스 256개마다 평균을 따로 추정한다. 클래스가 많아지면
(예: 16비트 변수 = 65,536 클래스) 클래스당 표본이 말라 학습이 무너진다.

`RLDAClassifier` 는 클래스마다 독립인 평균 대신, **비트의 선형 결합**으로 평균을 모형화한다.

$$ \mu_v \ \approx\ \sum_{b} c_b \cdot \mathrm{bit}_b(v) \ +\ c_0 $$

추정할 계수가 클래스 수가 아니라 **비트 수에 비례**하므로 표본이 훨씬 덜 든다.

함께 `RLDAInformationEstimator` 로 **모델이 실제로 얼마나 많은 정보를 담고 있는지**를
비트 단위로 잰다. "공격이 성공했다/실패했다" 보다 정량적인 평가 지표다.

| | |
|---|---|
| 학습 | `/profiling` |
| 하드웨어 | 불필요. `1.0.SNR` 선행 필요 |

In [1]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = "NanumGothic"
matplotlib.rcParams["axes.unicode_minus"] = False

# 공용 정의(AES 상수·데이터셋 로더)는 scalib_common.py 한 곳에만 둔다.
# 노트북마다 복사하면 값을 고칠 곳이 10군데가 되기 때문이다.
sys.path.insert(0, ".")
from scalib_common import SBOX, HW, AES_BLOCK, sbox_out, load_group, dataset_summary

import scalib
print("SCALib :", scalib.__version__)
print()
print(dataset_summary())

SCALib : 0.6.4

측정 조건
  created      2026-08-07T00:34:27
  cipher       AES-128-ECB (tiny-AES-c)
  platform     CW308_STM32F3
  ns           33172
  adc_mul      4
  adc_freq     29537976.0
  clk_hz       7384494.0
  gain_db      25.091743119266056
  trace_scale  32768.0

그룹
  /attack      10000 장 x 33172 샘플   키=fixed 평문=random
  /explore      5000 장 x 33172 샘플   키=random 평문=random
  /profiling   39000 장 x 33172 샘플   키=? 평문=?
  /tvla_fk      1000 장 x 33172 샘플   키=fixed 평문=fixed
  /tvla_rk      1000 장 x 33172 샘플   키=random 평문=fixed


In [2]:
from pathlib import Path

POI_PATH = Path("nb_output/poi.npz")
if not POI_PATH.is_file():
    raise FileNotFoundError(
        "nb_output/poi.npz 가 없다. 1.0.SNR.ipynb 를 먼저 실행한다.")
pz = np.load(POI_PATH)
poi, windows = pz["poi"], pz["windows"]
LO, HI = int(windows.min()), int(windows.max()) + 1
print("POI 구간 [%d, %d) — %d 샘플" % (LO, HI, HI - LO))

POI 구간 [6025, 6918) — 893 샘플


In [3]:
BYTE = 0
N_PROF = 60000        # RLDA 는 표본 요구가 적어 부분집합으로 충분하다

prof = load_group("profiling", n=N_PROF, samples=slice(LO, HI))
atk = load_group("attack", n=2000, samples=slice(LO, HI))
TRUE_KEY = np.array(atk["attrs"]["fixed_key"], dtype=np.uint8)

w = windows[BYTE] - LO
x_prof = sbox_out(prof["p"], prof["k"])[:, BYTE]
tr_prof = np.ascontiguousarray(prof["t"][:, w])
tr_atk = np.ascontiguousarray(atk["t"][:, w])
v_atk = SBOX[np.bitwise_xor(atk["p"][:, BYTE], TRUE_KEY[BYTE])]

print("학습 %s, 창 %d 샘플" % (tr_prof.shape, len(w)))

학습 (39000, 41), 창 41 샘플


---
## 1. `RLDAClassifier`

| 단계 | 코드 |
|:----:|------|
| 생성 | `rlda = RLDAClassifier(nb=8, p=4)` |
| 누적 | `rlda.fit_u(traces, x)` — `x` 는 **`(n, nv)` `uint64`** |
| 완료 | `rlda.solve()` |
| 판정 | `rlda.predict_proba(traces, var)` — 변수 번호를 지정한다 |

`nb` 는 변수의 **비트 수**다(클래스 수가 아니다). 바이트 변수면 8.

> **라벨 dtype 이 `uint64` 다.** 다른 API 는 `uint16` 인데 여기만 다르며,
> 학습은 2차원 `(n, nv)`, 뒤에 나올 정보량 추정기는 1차원 `(n,)` 을 받는다.

> **라벨 shape 규약이 API 마다 다르다.** 문서에 정리되어 있지 않고 실행해 봐야 알 수 있어
> 여기 적어 둔다. 틀리면 `ValueError: The classes array has N dimensions` 가 난다.
>
> | API | 라벨 shape | dtype |
> |-----|-----------|-------|
> | `Ttest` | `(n,)` | `uint16` |
> | `SNR`, `Cpa`, `MultiLDA` | `(n, nv)` | `uint16` |
> | `LDAClassifier` | `(n,)` | `uint16` |
> | `RLDAClassifier` | `(n, nv)` | `uint64` |
> | `RLDAInformationEstimator` | `(n,)` | `uint64` |
>
> 파형(`traces`)은 어디서나 `(n, ns)` `int16` 이다.

In [4]:
from scalib.modeling import RLDAClassifier
import time

P_DIM = 4
t0 = time.time()
rlda = RLDAClassifier(nb=8, p=P_DIM)
rlda.fit_u(tr_prof, x_prof.astype(np.uint64).reshape(-1, 1))   # (n, nv) uint64
rlda.solve()
print("RLDA 학습 %.1f 초" % (time.time() - t0))

pr_rlda = rlda.predict_proba(tr_atk, 0)     # 변수 0
print("predict_proba →", pr_rlda.shape)
print("정답 중간값의 평균 확률: %.4f (무작위 %.4f)"
      % (pr_rlda[np.arange(len(pr_rlda)), v_atk].mean(), 1 / 256))
print("정답이 1등인 비율     : %.1f%%" % (100 * (pr_rlda.argmax(axis=1) == v_atk).mean()))

RLDA 학습 0.0 초


predict_proba → (2000, 256)
정답 중간값의 평균 확률: 0.0226 (무작위 0.0039)
정답이 1등인 비율     : 4.2%


### LDA 와 나란히 — 학습 장수를 줄이면

RLDA 의 값어치는 **표본이 부족할 때** 드러난다. 클래스가 256개이므로 학습 1,000장이면
클래스당 4장꼴이고, 그 정도면 LDA 의 클래스 내 산포 행렬이 **특이(singular)** 해져
`solve()` 가 `ScalibError` 로 실패한다.

RLDA 는 클래스마다 평균을 따로 추정하지 않고 **비트의 선형 결합**으로 모형화하므로
추정할 계수가 훨씬 적고, 같은 조건에서 학습이 성립한다.

아래 표는 그 경계를 직접 보인다. LDA 가 실패하는 지점을 예외로 잡아 표시한다 —
**실패 자체가 결과**이기 때문이다.

In [5]:
from scalib.modeling import LDAClassifier

def try_lda(m):
    """LDA 학습을 시도한다. 표본 부족으로 실패하면 None 을 돌려준다."""
    try:
        l = LDAClassifier(nc=256, p=P_DIM)
        l.fit_u(tr_prof[:m], x_prof[:m].astype(np.uint16))
        l.solve()
        return (l.predict_proba(tr_atk).argmax(axis=1) == v_atk).mean()
    except Exception:
        return None

def try_rlda(m):
    r = RLDAClassifier(nb=8, p=P_DIM)
    r.fit_u(tr_prof[:m], x_prof[:m].astype(np.uint64).reshape(-1, 1))
    r.solve()
    return (r.predict_proba(tr_atk, 0).argmax(axis=1) == v_atk).mean()

n_avail = tr_prof.shape[0]
sizes = [m for m in (1000, 2560, 7500, 20000, 60000, 100000) if m <= n_avail]

print("학습 장수   클래스당      LDA           RLDA")
for m in sizes:
    la, ra = try_lda(m), try_rlda(m)
    la_s = "  학습 실패" if la is None else "%8.1f%%" % (100 * la)
    print("  %6d    %6.1f장    %-11s   %7.1f%%" % (m, m / 256, la_s, 100 * ra))
print()
print("→ 클래스당 표본이 얇을수록 LDA 가 먼저 무너진다. RLDA 는 그 구간에서도 동작한다.")
print("  이것이 클래스 수가 큰 변수(예: 16비트)에 RLDA 를 쓰는 이유다.")

학습 장수   클래스당      LDA           RLDA


    1000       3.9장      학습 실패           3.2%


    2560      10.0장        16.9%         4.5%


    7500      29.3장        18.1%         4.4%


   20000      78.1장        19.4%         4.2%

→ 클래스당 표본이 얇을수록 LDA 가 먼저 무너진다. RLDA 는 그 구간에서도 동작한다.
  이것이 클래스 수가 큰 변수(예: 16비트)에 RLDA 를 쓰는 이유다.


---
## 2. 정보량 추정 — `RLDAInformationEstimator`

"공격이 성공했는가" 는 이분법이다. **모델이 파형 한 장에서 몇 비트를 건지는가** 를 알면
구현끼리, 대책 적용 전후를 정량적으로 비교할 수 있다.

절차는 두 단계다.

1. `rlda.get_clustered_model(var, t)` — 계산을 감당할 수 있게 클래스를 묶은 간이 모델
2. `RLDAInformationEstimator(model, max_popped_classes)` → `fit_u` → `get_information()`

`get_information()` 은 **(하한, 상한)** 튜플을 준다. 최대 8비트(바이트 변수)이며,
클수록 그 모델이 파형에서 뽑아내는 정보가 많다는 뜻이다.

> **`get_deviation()` 은 호출하지 않는다.** scalib 0.6.4 에서 내부적으로 예외가 아닌 값을
> `raise` 해 `TypeError: exceptions must derive from BaseException` 가 난다(업스트림 버그).

In [6]:
from scalib.metrics import RLDAInformationEstimator

cm = rlda.get_clustered_model(0, t=0.5, max_clusters=10**7)
est = RLDAInformationEstimator(cm, max_popped_classes=512)
est.fit_u(tr_atk, v_atk.astype(np.uint64))      # 정보량 추정기는 라벨이 1차원
lo_i, hi_i = est.get_information()

print("정보량 추정: 하한 %.4f bit, 상한 %.4f bit  (최대 8 bit)" % (lo_i, hi_i))
print("→ 파형 한 장에서 이 모델이 건지는 정보량이다.")
print("   8 bit 면 중간값을 완전히 특정, 0 이면 아무것도 모르는 것과 같다.")

정보량 추정: 하한 1.7301 bit, 상한 1.7301 bit  (최대 8 bit)
→ 파형 한 장에서 이 모델이 건지는 정보량이다.
   8 bit 면 중간값을 완전히 특정, 0 이면 아무것도 모르는 것과 같다.


### 클러스터 임계 `t` 의 영향

`t` 가 작으면 클래스를 더 잘게 유지해 정확하지만 느리고, 크면 많이 묶어 빠르지만 거칠다.

In [7]:
print("  t     정보량(하한)   소요")
for t_val in (0.2, 0.5, 1.0, 2.0):
    t0 = time.time()
    c = rlda.get_clustered_model(0, t=t_val, max_clusters=10**7)
    e = RLDAInformationEstimator(c, max_popped_classes=512)
    e.fit_u(tr_atk, v_atk.astype(np.uint64))
    print("%5.1f    %8.4f bit    %.1f 초" % (t_val, e.get_information()[0], time.time() - t0))

  t     정보량(하한)   소요
  0.2      1.7301 bit    0.0 초
  0.5      1.7301 bit    0.0 초
  1.0      1.7301 bit    0.0 초
  2.0      1.7301 bit    0.0 초


---
## 3. 요약

| 항목 | 내용 |
|------|------|
| 목적 | 클래스가 많을 때도 견디는 **회귀 기반** 프로파일링, 그리고 **정보량** 정량화 |
| 모델 | `RLDAClassifier(nb, p)` → `fit_u(traces, x)` → `solve()` → `predict_proba(traces, var)` |
| `x` | 학습은 `(n, nv)` **`uint64`** |
| 정보량 | `get_clustered_model(var, t)` → `RLDAInformationEstimator(...)` → `get_information()` → (하한, 상한) 비트 |
| 주의 | `RLDAInformationEstimator.fit_u` 의 라벨은 **1차원** `uint64`. `get_deviation()` 은 0.6.4 에서 버그 |

### 실패 시 점검

1. `TypeError: argument 'label' ... cannot be converted` → 라벨 shape/dtype 을 위 표대로 맞춘다.
2. 정보량이 0 근처 → POI 나 학습 장수를 점검한다.

다음: `5.0.SASCA.ipynb` — 여러 중간값의 정보를 그래프로 합친다.